# Enriched Micro Benchmark (1% tasks, 100% agents)

Ноутбук запускает новые core-солверы (`MILP`, `GAP-VRP`, стох. версии) на enriched-датасете с предрасчитанной матрицей расстояний.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import sys
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'demo' else Path.cwd().resolve()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from flowopt.solvers.enriched import (
    solve_enriched_milp,
    solve_enriched_milp_stochastic,
    solve_enriched_batched_greedy,
    solve_enriched_stochastic_rr,
    solve_enriched_stochastic_grasp,
    solve_enriched_legacy_gap_vrp,
    solve_enriched_legacy_milp,
)


In [2]:
DATASET_PATH = REPO_ROOT / 'demo' / 'data' / 'object_mass_feasible_fullfleet' / 'container_full_split_all_agents' / 'sweeps_task_agent_5pct' / 'dataset_real_spb_clean_full_split_by_containers_all_agents_with_distances_t001_a100.json'

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Not found: {DATASET_PATH}. Build it with build_task_agent_sweep_with_constraints.py (1% tasks, 100% agents).'
    )

print('REPO_ROOT   :', REPO_ROOT)
print('DATASET_PATH:', DATASET_PATH)


REPO_ROOT   : /Users/igoreshka/Desktop/Optimization-of-flows
DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/demo/data/object_mass_feasible_fullfleet/container_full_split_all_agents/sweeps_task_agent_5pct/dataset_real_spb_clean_full_split_by_containers_all_agents_with_distances_t001_a100.json


In [3]:
MILP_TIME_LIMIT_SEC = 25
MILP_MAX_PAIRS_PER_TASK = 80

MILP_STOCH_TIME_BUDGET_SEC = 20.0
MILP_STOCH_MAX_STARTS = 8
MILP_STOCH_PER_START_SEC = 8
MILP_STOCH_MAX_PAIRS_PER_TASK = 50

GREEDY_TOP_K_AGENTS = 30
GREEDY_BALANCE_PENALTY = 0.02

STOCH_TIME_BUDGET_SEC = 10.0
STOCH_MAX_STARTS = 8

# Legacy pipelines are heavier on 1% dataset. Enable when needed.
RUN_LEGACY_GAP_VRP = False
RUN_LEGACY_MILP = False
LEGACY_GAP_ITER = 8
LEGACY_MILP_TIME_LIMIT_SEC = 30

results = []

results.append(
    solve_enriched_milp(
        dataset_path=DATASET_PATH,
        time_limit_sec=MILP_TIME_LIMIT_SEC,
        max_pairs_per_task=MILP_MAX_PAIRS_PER_TASK,
        use_repair=False,
        fallback_to_greedy=False,
    )
)

results.append(
    solve_enriched_milp_stochastic(
        dataset_path=DATASET_PATH,
        time_budget_sec=MILP_STOCH_TIME_BUDGET_SEC,
        max_starts=MILP_STOCH_MAX_STARTS,
        per_start_time_limit_sec=MILP_STOCH_PER_START_SEC,
        max_pairs_per_task=MILP_STOCH_MAX_PAIRS_PER_TASK,
        seed=42,
        use_repair=False,
        fallback_to_greedy=False,
    )
)

results.append(
    solve_enriched_batched_greedy(
        dataset_path=DATASET_PATH,
        top_k_agents=GREEDY_TOP_K_AGENTS,
        balance_penalty=GREEDY_BALANCE_PENALTY,
        random_seed=42,
    )
)

results.append(
    solve_enriched_stochastic_rr(
        dataset_path=DATASET_PATH,
        time_budget_sec=STOCH_TIME_BUDGET_SEC,
        max_starts=STOCH_MAX_STARTS,
        seed=42,
    )
)

results.append(
    solve_enriched_stochastic_grasp(
        dataset_path=DATASET_PATH,
        time_budget_sec=STOCH_TIME_BUDGET_SEC,
        max_starts=STOCH_MAX_STARTS,
        seed=42,
    )
)

if RUN_LEGACY_GAP_VRP:
    results.append(
        solve_enriched_legacy_gap_vrp(
            dataset_path=DATASET_PATH,
            step1_method='dataset',
            gap_iter=LEGACY_GAP_ITER,
            use_repair=True,
            show_progress=False,
            verbose=False,
        )
    )

if RUN_LEGACY_MILP:
    results.append(
        solve_enriched_legacy_milp(
            dataset_path=DATASET_PATH,
            time_limit_sec=LEGACY_MILP_TIME_LIMIT_SEC,
            unassigned_penalty=1e5,
            show_progress=False,
        )
    )

summary = pd.DataFrame([r.as_dict() for r in results])
summary = summary[[
    'algorithm', 'feasible', 'assigned_routes', 'assigned_trips', 'unassigned_tasks', 'active_agents',
    'transport_work_ton_km', 'total_km', 'deadhead_km', 'deadhead_share_pct', 'total_hours', 'runtime_sec', 'solver_error'
]]
summary.sort_values(by=['feasible', 'unassigned_tasks', 'total_km', 'runtime_sec'], ascending=[False, True, True, True]).reset_index(drop=True)


,algorithm,feasible,assigned_routes,assigned_trips,unassigned_tasks,active_agents,transport_work_ton_km,total_km,deadhead_km,deadhead_share_pct,total_hours,runtime_sec,solver_error
0,enriched_batched_greedy_v1,True,1240,913,0,525,2392.824,54829.789,39631.965,72.282,2518.408,1.311,None
1,enriched_stochastic_rr_v2,True,1240,913,0,525,2392.824,54829.789,39631.965,72.282,2517.969,10.463,None
2,enriched_stochastic_grasp_v2,True,1240,913,0,525,2392.824,54829.789,39631.965,72.282,2518.919,10.711,None
3,enriched_milp_stochastic_v1,False,1154,1154,86,472,2190.002,71087.049,51366.096,72.258,3151.978,26.552,None
4,enriched_milp_v2,False,1048,1048,192,293,2011.698,57252.782,41278.266,72.098,2575.065,26.178,None


In [4]:
checks_rows = []
for r in results:
    d = r.as_dict()
    checks = (d.get('details') or {}).get('checks') or {}
    checks_rows.append({
        'algorithm': d['algorithm'],
        'all_checks_ok': checks.get('all_checks_ok'),
        'daily_limits_ok': checks.get('daily_limits_ok'),
        'object_limits_ok': checks.get('object_limits_ok'),
        'compatibility_ok': checks.get('compatibility_ok'),
        'reachability_ok': checks.get('reachability_ok'),
        'all_tasks_assigned': checks.get('all_tasks_assigned'),
        'overflow_km_agents': checks.get('overflow_km_agents'),
        'overflow_hours_agents': checks.get('overflow_hours_agents'),
        'object_mass_violations': checks.get('object_mass_violations'),
        'object_volume_violations': checks.get('object_volume_violations'),
        'incompatible_assignments': checks.get('incompatible_assignments'),
        'unreachable_assignments': checks.get('unreachable_assignments'),
        'metric_task_space': checks.get('metric_task_space'),
    })

pd.DataFrame(checks_rows)


,algorithm,all_checks_ok,daily_limits_ok,object_limits_ok,compatibility_ok,reachability_ok,all_tasks_assigned,overflow_km_agents,overflow_hours_agents,object_mass_violations,object_volume_violations,incompatible_assignments,unreachable_assignments,metric_task_space
0,enriched_milp_v2,False,True,True,True,True,False,0,0,0,0,0,0,dataset_reference
1,enriched_milp_stochastic_v1,False,True,True,True,True,False,0,0,0,0,0,0,dataset_reference
2,enriched_batched_greedy_v1,True,True,True,True,True,True,0,0,0,0,0,0,dataset_reference
3,enriched_stochastic_rr_v2,True,True,True,True,True,True,0,0,0,0,0,0,dataset_reference
4,enriched_stochastic_grasp_v2,True,True,True,True,True,True,0,0,0,0,0,0,dataset_reference


In [5]:
OUT_DIR = REPO_ROOT / 'demo' / 'local' / 'enriched_micro_1pct'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = OUT_DIR / f'benchmark_{ts}.json'

payload = {
    'dataset_path': str(DATASET_PATH),
    'created_at': ts,
    'results': [r.as_dict() for r in results],
}
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', out_path)


Saved: /Users/igoreshka/Desktop/Optimization-of-flows/demo/local/enriched_micro_1pct/benchmark_20260504_204824.json
